# Z3 (C# / .NET) — Ordonnancement de taches (Job-Shop Scheduling)

**Twin C# de `Z3-08-Ordonnancement-Python.ipynb`** (parite .NET : meme instance, memes verdicts, documentation en francais, API `Microsoft.Z3`).

**Navigation** : [Index](README.md) | [Index SMT](../README.md) | [Index SymbolicAI](../../README.md) | [<< 06 - Advanced Optimization (C#)](Z3-06-Advanced-Optimization-CSharp.ipynb) | [09 - Enigme d'Einstein (Python) >>](Z3-09-Enigme-Einstein-Python.ipynb)

## Le probleme

Le **job-shop scheduling** est un probleme canonique d'optimisation combinatoire : on dispose de $m$ machines et de $n$ jobs, chaque job etant une **sequence d'operations** (une par machine) de durees donnees. On veut trouver un ordonnancement (date de debut de chaque operation) qui :

1. **respecte l'ordre** des operations de chaque job (precedence),
2. **n'utilise chaque machine qu'une fois a la fois** (exclusion disjonctive),
3. **minimise le makespan** $C_{\max}$ (la date de fin de la derniere operation).

Ce probleme est **NP-difficile** : il n'existe pas d'algorithme polynomial connu, et une approche gloutonne (heuristique) rate souvent l'optimum. C'est typiquement la ou un solveur SMT comme Z3 **brille** : on decrit les contraintes (precedence + exclusion) et l'objectif (minimiser $C_{\max}$), et `Optimize` trouve l'ordonnancement optimal.

**Ce carnet montre** : (a) une heuristique gloutonne FIFO sous-optimale, (b) la modelisation Z3 avec la contrainte disjonctive `MkOr(...)`, (c) l'optimum trouve par `MkMinimize`, et (d) un diagramme de Gantt comparant les deux. C'est le portage .NET du carnet Python [Z3-08-Ordonnancement-Python.ipynb](Z3-08-Ordonnancement-Python.ipynb), lui-meme porte de l'exemple C# [11_Job_Shop_Scheduling](../Z3-Linq2Z3/11_Job_Shop_Scheduling.ipynb) de la serie soeur Z3.Linq.

## Source

Ce carnet s'inscrit dans la serie ancree par **Yurichev, Dennis — *SAT/SMT by Example* (edition publique du 12 fevrier 2023, 666 pages)**, §22.8 Job Shop Scheduling (p. 633), §22.8.1 My first version (p. 633) et §22.8.2 MaxSMT version by Chad Brewbaker (p. 635) — le probleme est pose a l'identique (jobs, machines, operations, `makespan` minimise, diagramme de Gantt), avec les instances 3x3 puis 10 machines. Le carnet reprend l'instance 3 jobs x 2 machines et la comparaison glouton vs optimum. Gisement : `G:\Mon Drive\MyIA\IA\Bibliographie IA\Symbolic\Satisfiability and SMT Solvers\2023 - SAT_SMT_by_example.pdf` (16,45 Mo, magic bytes `%PDF`). Pages citees d'apres la pagination imprimee du fichier depose. Voir [Z3-API/README.md — Ouvrage de reference](README.md#ouvrage-de-reference) pour l'index chapitre de la serie.

In [1]:
// References : Microsoft.Z3 (solveur SMT, binding officiel du moteur C++) + SVG inline pour le Gantt.
#r "nuget: Microsoft.Z3"
#load "Z3NativeLoader.cs"
using Microsoft.Z3;
using System;
using System.Collections.Generic;
using System.Linq;
using System.Text;

// Bibliotheque native de Z3 hors Windows et macOS Intel (voir Z3NativeLoader.cs)
Z3NativeLoader.Register(typeof(Context).Assembly);

// Contexte Z3 partage par toutes les cellules (meme moteur C++ que z3-solver Python)
var ctx = new Context();
Console.WriteLine($"Moteur : Microsoft.Z3 {Microsoft.Z3.Version.FullVersion}");
Console.WriteLine("Ordonnancement : Optimize.MkMinimize, contrainte disjonctive MkOr.");

Installing Packages Microsoft.Z3

Moteur : Microsoft.Z3 Z3 4.12.2.0


Ordonnancement : Optimize.MkMinimize, contrainte disjonctive MkOr.


## Instance de reference : 3 jobs x 2 machines

On encode l'instance comme un dictionnaire : chaque job est la liste ordonnee de ses operations, chaque operation etant un couple `(machine, duree)`.

| Job | Operation 1 | Operation 2 |
|-----|-------------|-------------|
| J0  | M0 (3h)     | M1 (3h)     |
| J1  | M1 (2h)     | M0 (2h)     |
| J2  | M0 (2h)     | M1 (2h)     |

Notez que J0 et J2 commencent par M0, mais J1 commence par M1 : les jobs n'ont pas tous le meme parcours.

In [2]:
// Instance : 3 jobs x 2 machines (M0, M1).
// Format : jobs[j] = tableau ordonne d'operations (machine, duree).
string[] MACHINES = { "M0", "M1" };
var jobs = new Dictionary<string, (string m, int d)[]>
{
    ["J0"] = new[] { ("M0", 3), ("M1", 3) },  // 3h sur M0 puis 3h sur M1
    ["J1"] = new[] { ("M1", 2), ("M0", 2) },  // 2h sur M1 puis 2h sur M0
    ["J2"] = new[] { ("M0", 2), ("M1", 2) },  // 2h sur M0 puis 2h sur M1
};
string[] JOB_NAMES = jobs.Keys.ToArray();

Console.WriteLine($"Instance : {JOB_NAMES.Length} jobs x {MACHINES.Length} machines");
foreach (var j in JOB_NAMES)
{
    string parcours = string.Join(" puis ", jobs[j].Select(op => $"{op.m}({op.d}h)"));
    Console.WriteLine($"  {j} : {parcours}");
}

Instance : 3 jobs x 2 machines


  J0 : M0(3h) puis M1(3h)


  J1 : M1(2h) puis M0(2h)


  J2 : M0(2h) puis M1(2h)


## Pourquoi ce probleme est-il difficile ?

Le job-shop scheduling est **NP-difficile** (Garey & Johnson, 1976) : des que le nombre de jobs et de machines croit, l'espace des ordonnancements possibles explose, car il faut choisir un **ordre des operations sur chaque machine** parmi tous les ordres compatibles. Notre instance 3 jobs x 2 machines reste completement lisible — c'est le but pedagogique — mais la meme structure a 10 jobs x 10 machines defie les approches exactes depuis les annees 1960 (l'instance canonique FT10 de Fisher & Thompson n'a ete close qu'en 1989).

Deux bornes encadrent deja l'espace des solutions :

- **Borne inferieure de charge** : chaque machine doit absorber la somme des durees des operations qui la traversent, soit 7 h pour M0 (3 + 2 + 2) et 7 h pour M1 (3 + 2 + 2). Aucun ordonnancement ne peut finir avant t = 7.
- **Horizon sequentiel** : si tout etait execute a la suite, la date de fin serait la somme de toutes les durees : 6 + 4 + 4 = **14 h**. C'est exactement la valeur que donnera le glouton ci-dessous — et la borne superieure que nous fournirons a Z3.

La question devient : peut-on descendre sous 14 h en **entrelacant** les jobs ? C'est precisement ce que la contrainte disjonctive va exprimer.

## Approche gloutonne : ordonnancement FIFO

Avant Z3, essayons une **heuristique simple** : on schedule les jobs dans l'ordre (J0, puis J1, puis J2), et pour chaque job on place chaque operation **le plus tot possible** (des que la machine est libre et que l'operation precedente du job est terminee).

Cette strategie FIFO est rapide mais **myope** : elle ne regarde pas plus loin que le job courant, et peut laisser une machine inutilisee la ou un autre ordre aurait mieux equilibre la charge.

In [3]:
// Ordonnancement glouton : chaque job dans l'ordre, chaque operation placee au plus tot.
(Dictionary<string, List<(string m, int s, int d)>> schedule, int cmax) GreedyFifo(
    Dictionary<string, (string m, int d)[]> jobs, string[] machineNames)
{
    // Disponibilite de chaque machine
    var machineFree = machineNames.ToDictionary(m => m, m => 0);
    var schedule = new Dictionary<string, List<(string m, int s, int d)>>();
    foreach (var j in jobs.Keys)
    {
        var ops = new List<(string m, int s, int d)>();
        foreach (var (m, d) in jobs[j])
        {
            int jobReady = ops.Count > 0 ? ops[^1].s + ops[^1].d : 0;  // fin de l'operation precedente
            int start = Math.Max(machineFree[m], jobReady);
            ops.Add((m, start, d));
            machineFree[m] = start + d;
        }
        schedule[j] = ops;
    }
    int cmax = schedule.Values.SelectMany(o => o).Max(op => op.s + op.d);
    return (schedule, cmax);
}

var (greedySchedule, greedyCmax) = GreedyFifo(jobs, MACHINES);
Console.WriteLine($"Glouton FIFO : Cmax = {greedyCmax}h");
foreach (var j in JOB_NAMES)
{
    string ops = string.Join(", ", greedySchedule[j].Select(op => $"{op.m}[{op.s},{op.s + op.d}]"));
    Console.WriteLine($"  {j} : {ops}");
}

Glouton FIFO : Cmax = 14h


  J0 : M0[0,3], M1[3,6]


  J1 : M1[6,8], M0[8,10]


  J2 : M0[10,12], M1[12,14]


## Modelisation Z3 : variables et contraintes

Le passage au declaratif. On introduit une variable entiere $s_{j,k}$ = date de debut de l'operation $k$ du job $j$, plus une variable $C_{\max}$ pour le makespan. Les contraintes se traduisent directement :

1. **Precedence intra-job** : l'operation $k$ commence apres la fin de l'operation $k{-}1$ :  $s_{j,k} \ge s_{j,k-1} + d_{j,k-1}$.
2. **Exclusion disjonctive** (le coeur du modele) : pour deux operations $a$ et $b$ sur la **meme** machine, l'une doit finir avant que l'autre commence :  $s_a + d_a \le s_b$ **OU** $s_b + d_b \le s_a$. C'est cette disjonction — `ctx.MkOr(...)` en C# — (non-lineaire, difficile pour un simple parcours) qui fait toute la valeur du solveur.
3. **Objectif** : minimiser $C_{\max} = \max_{j,k}(s_{j,k} + d_{j,k})$, via `opt.MkMinimize(cmax)`.

La borne superieure sure $C_{\max} \le$ somme de toutes les durees (cas ou tout serait sequentialise) garantit un domaine fini au solveur.

In [4]:
// Modelise le job-shop en Z3 et minimise le makespan (Cmax).
(Dictionary<string, List<(string m, int s, int d)>> schedule, int cmax) SolveJobShop(
    Dictionary<string, (string m, int d)[]> jobs, string[] machineNames)
{
    var opt = ctx.MkOptimize();
    var jobNames = jobs.Keys.ToArray();
    int horizon = jobs.Values.SelectMany(o => o).Sum(op => op.d);  // borne superieure sure

    // Variables de debut : start[j][k]
    var start = jobNames.ToDictionary(
        j => j,
        j => Enumerable.Range(0, jobs[j].Length).Select(k => (IntExpr)ctx.MkIntConst($"s_{j}_{k}")).ToArray());
    var cmax = (IntExpr)ctx.MkIntConst("cmax");

    // (1) Domaine + precedence intra-job + borne makespan
    foreach (var j in jobNames)
    {
        for (int k = 0; k < jobs[j].Length; k++)
        {
            int d = jobs[j][k].d;
            opt.Assert(ctx.MkGe(start[j][k], ctx.MkInt(0)));
            opt.Assert(ctx.MkLe(ctx.MkAdd(start[j][k], ctx.MkInt(d)), cmax));
            if (k > 0)
            {
                int pd = jobs[j][k - 1].d;
                opt.Assert(ctx.MkGe(start[j][k], ctx.MkAdd(start[j][k - 1], ctx.MkInt(pd))));
            }
        }
    }

    // (2) Exclusion disjonctive : deux operations sur la MEME machine ne se chevauchent pas
    foreach (var m in machineNames)
    {
        var opsOnM = jobNames
            .SelectMany(j => Enumerable.Range(0, jobs[j].Length).Select(k => (j, k, d: jobs[j][k].d)))
            .Where(op => jobs[op.j][op.k].m == m)
            .ToArray();
        for (int a = 0; a < opsOnM.Length; a++)
        {
            for (int b = a + 1; b < opsOnM.Length; b++)
            {
                var (ja, ka, da) = opsOnM[a];
                var (jb, kb, db) = opsOnM[b];
                opt.Assert(ctx.MkOr(
                    ctx.MkLe(ctx.MkAdd(start[ja][ka], ctx.MkInt(da)), start[jb][kb]),
                    ctx.MkLe(ctx.MkAdd(start[jb][kb], ctx.MkInt(db)), start[ja][ka])));
            }
        }
    }

    // (3) Objectif : minimiser le makespan
    opt.Assert(ctx.MkLe(cmax, ctx.MkInt(horizon)));
    opt.MkMinimize(cmax);
    if (opt.Check() != Status.SATISFIABLE)
        throw new InvalidOperationException("Instance insatisfiable");
    var model = opt.Model;
    var schedule = new Dictionary<string, List<(string m, int s, int d)>>();
    foreach (var j in jobNames)
    {
        schedule[j] = Enumerable.Range(0, jobs[j].Length)
            .Select(k => (m: jobs[j][k].m,
                          s: (int)((IntNum)model.Evaluate(start[j][k])).Int64,
                          d: jobs[j][k].d))
            .ToList();
    }
    return (schedule, (int)((IntNum)model.Evaluate(cmax)).Int64);
}

var (optSchedule, optCmax) = SolveJobShop(jobs, MACHINES);
Console.WriteLine($"Z3 optimal : Cmax = {optCmax}h  (glouton = {greedyCmax}h, gain = {greedyCmax - optCmax}h)");

Z3 optimal : Cmax = 8h  (glouton = 14h, gain = 6h)


## Anatomie du modele genere

Avant de lire le resultat, comptons ce que Z3 manipule reellement pour cette instance :

| Element | Quantite | Detail |
|---|---|---|
| Variables entieres | 7 | $s_{j,k}$ pour 6 operations + $C_{\max}$ |
| Precedences intra-job | 3 | une par job (l'operation 2 suit l'operation 1) |
| Bornes de domaine | 6 | $s_{j,k} \ge 0$, plus $s_{j,k} + d_{j,k} \le C_{\max}$ |
| Contraintes disjonctives | 6 | 3 paires sur M0, 3 paires sur M1 |

Les paires disjonctives se repartissent ainsi : M0 porte les operations J0-1, J1-2 et J2-1, soit $\binom{3}{2} = 3$ paires ; M1 porte J0-2, J1-1 et J2-2, soit 3 paires egalement. Chaque paire produit un `MkOr(...)` : l'une des deux operations finit avant que l'autre commence. Ces 6 disjonctions sont le coeur combinatoire du modele — c'est elles qui codent le choix de l'ordre sur chaque machine, sans jamais l'enumerer.

Notez enfin le role d'`Optimize` (et non `Solver`) : la ou un `Solver` renverrait le **premier** ordonnancement faisable trouve, `MkMinimize(cmax)` exige le makespan **minimal**, avec garantie d'optimalite — le solveur prouve qu'aucune valeur inferieure n'est satisfiable. C'est une preuve, pas une heuristique.

## Resultat optimal

Le solveur trouve l'ordonnancement de makespan **minimal**. Comparons le detail operation par operation : la ou le glouton sequentialisait trop, Z3 entrelace les jobs (une operation d'un job pendant le creux d'un autre) pour reduire le temps total.

In [5]:
// Comparaison cote a cote : chaque job affiche sa chaine glouton puis sa chaine Z3.
// Une operation se lit m<debut,fin> : machine, creneau de debut, creneau de fin.
Console.WriteLine("Comparaison glouton vs Z3 optimal :");
Console.WriteLine();
foreach (var j in JOB_NAMES)
{
    // Les deux ordonnancements sont projetes dans le meme format pour se lire en colonne.
    string g = string.Join(", ", greedySchedule[j].Select(op => $"{op.m}[{op.s},{op.s + op.d}]"));
    string o = string.Join(", ", optSchedule[j].Select(op => $"{op.m}[{op.s},{op.s + op.d}]"));
    Console.WriteLine($"  {j}");
    Console.WriteLine($"    glouton : {g}");
    Console.WriteLine($"    optimal : {o}");
}
Console.WriteLine();
// Le gain est rapporte en heures et en pourcentage du makespan glouton.
Console.WriteLine($"Makespan : glouton {greedyCmax}h -> optimal {optCmax}h (gain de {greedyCmax - optCmax}h, {100.0 * (greedyCmax - optCmax) / greedyCmax:F0}%)");

Comparaison glouton vs Z3 optimal :


  J0


    glouton : M0[0,3], M1[3,6]


    optimal : M0[0,3], M1[3,6]


  J1


    glouton : M1[6,8], M0[8,10]


    optimal : M1[1,3], M0[6,8]


  J2


    glouton : M0[10,12], M1[12,14]


    optimal : M0[4,6], M1[6,8]


Makespan : glouton 14h -> optimal 8h (gain de 6h, 43%)


## Lecture de la comparaison : d'ou vient l'ecart 14 h -> 8 h ?

La sortie ci-dessus se lit ligne par ligne, et tout se joue sur **J1**. J0 est identique dans les deux ordonnancements (M0[0,3] puis M1[3,6]) : c'est le premier job traite, le glouton ne peut pas se tromper. Mais observez J1 :

- **Glouton** : M1[6,8] puis M0[8,10]. Pourquoi attendre t = 6 pour sa premiere operation ? Parce que le glouton traite les jobs **les uns apres les autres** : quand vient le tour de J1, la plage M1[3,6] est deja reservee par J0, et le glouton ne place jamais une operation **dans un trou** laisse avant un placement precedent. La fenetre M1[0,3] — parfaitement libre a ce moment-la — reste inutilisee.
- **Optimal** : M1[1,3] puis M0[6,8]. Z3 n'a aucune notion d'ordre de traitement : il place les 6 operations simultanement dans le temps, et remplit la fenetre M1[1,3] que le glouton a laissee vide.

J2 subit le meme effet en cascade : recule a M0[10,12]/M1[12,14] par le glouton, il est avance a M0[4,6]/M1[6,8] par l'optimal. Le glouton aboutit a 14 h — exactement l'horizon sequentiel, la pire borne possible — tandis que l'optimal gagne 6 h (**43 %**), une heure seulement au-dessus de la borne de charge (7 h).

## Visualisation : diagramme de Gantt

Le diagramme de Gantt rend visible la difference : chaque ligne est une machine, chaque barre coloree une operation. Le glouton (haut) laisse des trous ; l'optimal Z3 (bas) tasse les operations pour finir plus tot.

Cote .NET, le Gantt est un **SVG inline statique**, sans dependance graphique : il est produit par le helper canon `SvgChartHelper` du depot (conteneur `SvgChart` + formateur `text/html`, canon #6942) et s'affiche tel quel sur GitHub, nbviewer et hors ligne — meme principe que les figures des autres carnets .NET de la serie.

In [6]:
#load "../../../Probas/Infer/SvgChartHelper.cs"
using System.Globalization;
using System.Linq;

// Couleurs par job (identiques au jumeau Python)
string[] GANTT_COLORS = { "#4C72B0", "#DD8452", "#55A868", "#C44E52" };

// Gantt : une ligne par machine, une barre par operation, couleur = job.
// SvgChartHelper fournit le conteneur `SvgChart` et son formateur text/html ; il n'a pas de
// primitive a barres horizontales decalees, donc le SVG est genere ici et confie au meme formateur.
// Le formateur est enregistre par le constructeur STATIQUE du helper : on l'amorce explicitement
// (resultat ignore), sinon un `new SvgChart(...)` isolé s'afficherait via l'inspecteur d'objet.
_ = SvgChartHelper.Bar("amorce", new[] { "0" }, new[] { 0.0 });

SvgChart BuildGantt(Dictionary<string, List<(string m, int s, int d)>> schedule,
                    string title, string[] machineNames, int xmax, int width = 820)
{
    int rowH = 48, padL = 60, padT = 48, padR = 20, padB = 38;
    int height = padT + rowH * machineNames.Length + padB;
    int plotW = width - padL - padR;
    string F(double v) => v.ToString("F1", CultureInfo.InvariantCulture);
    double X(double t) => padL + t * plotW / (double)xmax;

    var sb = new StringBuilder();
    sb.Append($"<svg xmlns='http://www.w3.org/2000/svg' width='{width}' height='{height}' font-family='sans-serif' font-size='11'>");
    sb.Append("<rect width='100%' height='100%' fill='white'/>");
    sb.Append($"<text x='{F(width / 2.0)}' y='20' text-anchor='middle' font-size='13' font-weight='bold' fill='#333333'>{title}</text>");
    // Grille verticale + graduations horaires
    for (int t = 0; t <= xmax; t++)
    {
        sb.Append($"<line x1='{F(X(t))}' y1='{padT}' x2='{F(X(t))}' y2='{padT + rowH * machineNames.Length}' stroke='#E5E5E5'/>");
        sb.Append($"<text x='{F(X(t))}' y='{padT + rowH * machineNames.Length + 16}' text-anchor='middle' fill='#333333'>{t}</text>");
    }
    sb.Append($"<text x='{F(padL + plotW / 2.0)}' y='{height - 6}' text-anchor='middle' fill='#333333'>Temps (heures)</text>");
    // Une rangee par machine
    for (int mi = 0; mi < machineNames.Length; mi++)
    {
        string machine = machineNames[mi];
        double yBar = padT + mi * rowH + 11;
        sb.Append($"<text x='{padL - 8}' y='{F(yBar + 17)}' text-anchor='end' fill='#333333'>{machine}</text>");
        for (int ji = 0; ji < schedule.Count; ji++)
        {
            string job = schedule.ElementAt(ji).Key;
            foreach (var (mach, s, d) in schedule[job])
            {
                if (mach != machine) continue;
                double x1 = X(s), x2 = X(s + d);
                sb.Append($"<rect x='{F(x1)}' y='{F(yBar)}' width='{F(x2 - x1)}' height='26' fill='{GANTT_COLORS[ji % GANTT_COLORS.Length]}' stroke='black' stroke-width='0.5'/>");
                sb.Append($"<text x='{F((x1 + x2) / 2.0)}' y='{F(yBar + 17)}' text-anchor='middle' font-weight='bold' fill='white'>{job} {d}h</text>");
            }
        }
    }
    // Legende : une pastille par job
    double lx = padL, ly = 34;
    for (int ji = 0; ji < schedule.Count; ji++)
    {
        sb.Append($"<rect x='{F(lx)}' y='{ly - 9}' width='12' height='9' fill='{GANTT_COLORS[ji % GANTT_COLORS.Length]}'/>");
        sb.Append($"<text x='{F(lx + 16)}' y='{ly}' fill='#333333'>{schedule.ElementAt(ji).Key}</text>");
        lx += 56;
    }
    sb.Append("</svg>");
    return new SvgChart(sb.ToString());
}

int xRange = Math.Max(greedyCmax, optCmax);
display(BuildGantt(greedySchedule, $"Glouton FIFO (Cmax = {greedyCmax}h)", MACHINES, xRange));
display(BuildGantt(optSchedule, $"Z3 optimal (Cmax = {optCmax}h)", MACHINES, xRange));
Console.WriteLine("Diagramme de Gantt : glouton (haut) vs optimal (bas).");

Glouton FIFO (Cmax = 14h) 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 Temps (heures) M0 J0 3h J1 2h J2 2h M1 J0 3h J1 2h J2 2h J0 J1 J2

Z3 optimal (Cmax = 8h) 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 Temps (heures) M0 J0 3h J1 2h J2 2h M1 J0 3h J1 2h J2 2h J0 J1 J2

Diagramme de Gantt : glouton (haut) vs optimal (bas).


## Lecture du diagramme

Les deux Gantt ci-dessus rendent visibles les defauts du glouton. Sur le Gantt optimal (bas), chaque machine ne reste **inoccupee qu'une seule heure** : M0 sur [3,4] et M1 sur [0,1]. Les trois jobs s'entrelacent — la barre J1 sur M1 occupe [1,3] pendant que J0 travaille sur M0, puis les roles s'echangent. Le makespan de 8 h correspond a la fin simultanee des deux dernieres operations (J1 sur M0 et J2 sur M1).

Un point merite reflexion : la borne inferieure de charge (7 h) n'est **pas atteinte**. Un ordonnancement a 7 h remplirait chaque machine sans aucune interruption — le solveur a explore cette possibilite et prouve qu'elle viole necessairement une precedence ou une disjonction. C'est l'apport decisif d'`Optimize` : la valeur 8 h est **certifiee minimale**. Une metaheuristique (recuit simule, colonies de fourmis) pourrait trouver 8 h aussi, mais ne pourrait jamais certifier qu'il n'existe pas de solution a 7 h.

Les exercices suivants prolongent exactement ce geste : ajouter un job (l'espace de recherche grandit), une deadline (contrainte dure), une machine (nouvelles disjonctions).

## Exercices

Les trois exercices suivants vous font manipuler le modele : etendre l'instance (1), ajouter une contrainte metier de deadline (2), et monter en dimension avec une troisieme machine (3). Chaque stub reutilise `SolveJobShop` defini plus haut.

### Exercice 1 -- Ajout d'un quatrieme job

**Objectif** : ajouter un job `J3 = [("M1", 1), ("M0", 3)]` (1h sur M1 puis 3h sur M0) et re-resoudre. Observez comment le makespan optimal augmente (ou non) et comment Z3 re-entrelace les operations.

**Indices** : copiez le dictionnaire `jobs` dans `jobs4` (constructeur de copie du dictionnaire, les tableaux d'operations sont immuables en pratique) ; ajoutez l'entree J3 ; appelez `SolveJobShop(jobs4, MACHINES)`.

In [7]:
// Exercice 1 : ajoutez un quatrieme job J3 et re-resolvez.
// Etape 1 : var jobs4 = new Dictionary<string, (string m, int d)[]>(jobs);
//            jobs4["J3"] = new[] { ("M1", 1), ("M0", 3) };
// Etape 2 : resolvez via SolveJobShop(jobs4, MACHINES).
// Etape 3 : comparez le nouveau Cmax a l'optimum 3 jobs (optCmax).

(Dictionary<string, List<(string m, int s, int d)>> schedule, int cmax)? Resoudre4Jobs()
{
    // TODO etudiant : construire jobs4 et appeler SolveJobShop
    return null;
}

Console.WriteLine("Exercice 1 a completer : ajouter un 4eme job et mesurer le nouveau Cmax optimal.");

Exercice 1 a completer : ajouter un 4eme job et mesurer le nouveau Cmax optimal.


### Exercice 2 -- Contrainte d'urgence (deadline)

**Objectif** : le job `J1` vient de passer urgent : imposez qu'il termine avant $t = 6$ (dans l'ordonnancement optimal exhibe plus haut, J1 termine a $t = 8$). Ajoutez cette contrainte au modele et verifiez que le makespan optimal augmente — mesure attendue au solveur : Cmax passe de 8 a 9, la deadline force un ordonnancement moins efficace.

**Indices** : la fin de J1 est `start["J1"][^1] + jobs["J1"][^1].d` ; ajoutez `opt.Assert(... <= 6)` avant `opt.MkMinimize`. Modifiez `SolveJobShop` ou creez une variante.

In [8]:
// Exercice 2 : ajoutez une deadline (J1 doit terminer avant t = 6).
// Indice : opt.Assert(ctx.MkLe(ctx.MkAdd(start["J1"][^1], ctx.MkInt(jobs["J1"][^1].d)), ctx.MkInt(6)))
//          avant opt.MkMinimize(cmax).
// Etape 1 : copiez SolveJobShop en SolveJobShopDeadline(jobs, machineNames, deadlineJ1 = 6).
// Etape 2 : resolvez et comparez le Cmax a optCmax (sans deadline).

(Dictionary<string, List<(string m, int s, int d)>> schedule, int cmax)? SolveJobShopDeadline(
    Dictionary<string, (string m, int d)[]> jobs, string[] machineNames, int deadlineJ1 = 6)
{
    // TODO etudiant : modeliser, ajouter la deadline sur J1, minimiser
    return null;
}

Console.WriteLine("Exercice 2 a completer : mesurer le surcout d'une deadline sur J1.");

Exercice 2 a completer : mesurer le surcout d'une deadline sur J1.


### Exercice 3 -- Extension a 3 machines

**Objectif** : etendre l'instance a 3 machines (M0, M1, M2) avec des jobs a 3 operations chacun, et verifier que `SolveJobShop` generalise sans modification de code (le modele est parametre par les donnees).

**Indices** : definissez `MACHINES_3 = new[] { "M0", "M1", "M2" }` et des `jobs3` ou chaque job a 3 operations sur des machines distinctes ; la fonction `SolveJobShop` gere deja le cas general.

In [9]:
// Exercice 3 : etendez l'instance a 3 machines, 3 operations par job.
// Etape 1 : var machines3 = new[] { "M0", "M1", "M2" };
//            var jobs3 = new Dictionary<string, (string m, int d)[]> { ... 3 operations par job ... };
// Etape 2 : resolvez via SolveJobShop(jobs3, machines3).
// Question : le temps de resolution grimpe-t-il avec 3 machines ? Mesurez-le (Stopwatch).

(Dictionary<string, List<(string m, int s, int d)>> schedule, int cmax)? Resoudre3Machines()
{
    // TODO etudiant : definir jobs3 + machines3 et appeler SolveJobShop
    return null;
}

Console.WriteLine("Exercice 3 a completer : generaliser a 3 machines et observer le temps de resolution.");

Exercice 3 a completer : generaliser a 3 machines et observer le temps de resolution.


## Conclusion

Le job-shop scheduling illustre le gain du **declaratif** sur une classe de problemes ou l'heuristique gloutonne est structurellement sous-optimale :

- La **contrainte disjonctive** `MkOr(s_a + d_a <= s_b, s_b + d_b <= s_a)` (exclusion mutuelle sur une machine) est difficile a coder en imperatif sans enumerer tous les ordres possibles ; Z3 la traite nativement.
- `MkMinimize(cmax)` trouve le makespan **optimal** (prouvablement minimal), la ou FIFO se contente d'un ordonnancement realisable mais myope.
- Le modele est **parametre par les donnees** : ajouter un job, une machine ou une deadline se fait en modifiant l'instance ou en ajoutant une contrainte, sans re-ecrire l'algorithme de resolution.

**Limites** : sur de tres grandes instances (dizaines de jobs x dizaines de machines), le solveur peut devenir lent — c'est le prix de l'optimalite garantie. Dans ce cas, on combine souvent Z3 avec des heuristiques (fournir un ordonnancement glouton comme borne superieure aide le solveur a pruner).

Suite logique : le [jumeau Python](Z3-08-Ordonnancement-Python.ipynb) pour la version `pyz3`, la [serie Z3-API](README.md) pour la carte des carnets, ou la version [Z3.Linq](../Z3-Linq2Z3/README.md) qui aborde le meme probleme via une couche declarative LINQ.